In [21]:
# Install necessary Python libraries using pip
!pip install crewai pandas numpy requests ollama python-dotenv

# Importing essential libraries
import pandas as pd                     # For data manipulation and analysis
import numpy as np                      # For numerical computations
import json                             # To work with JSON data
import random                           # For generating random values
from datetime import datetime, timedelta  # For handling date and time operations
from typing import Dict, List, Any      # For type hinting
import ollama                           # For interacting with the Ollama LLM API
from crewai import Agent, Task, Crew, Process  # Core components from CrewAI
from crewai.tools import BaseTool       # Base class for custom tools in CrewAI
import warnings                         # To suppress warning messages
warnings.filterwarnings('ignore')       # Ignores any warning outputs

# Confirm successful installation and setup
print("All dependencies installed successfully!")

# Attempt to send a test message to the Ollama LLM to confirm it is working
try:
    response = ollama.chat(
        model='llama3.2',  # Specify the model name
        messages=[{'role': 'user', 'content': 'Hello, are you working?'}]  # Input prompt
    )
    print("Ollama connection successful!")  # Success message
    print(f"Response: {response['message']['content'][:50]}...")  # Display the first 50 characters of the response
except Exception as e:
    # If the connection fails, print the error message
    print(f"Ollama connection failed: {e}")
    print("Please ensure Ollama is running with: ollama serve")  # Help message for troubleshooting

























All dependencies installed successfully!
Ollama connection successful!
Response: I'm here and ready to help. What can I assist you ...


In [23]:
# Import necessary libraries
import pandas as pd                    # For handling Excel and tabular data
import numpy as np                     # For numerical operations
import json                            # To handle JSON serialization
from datetime import datetime, timedelta  # To manage and manipulate dates

class ExcelFrappeDB:
    """Class to handle reading business context data from Excel and matching it for given inputs."""

    def __init__(self, excel_path: str):
        # Initialize with Excel file path and load the data
        self.excel_path = excel_path
        self.load_data_from_excel()

    def load_data_from_excel(self):
        # Read Excel data into a DataFrame
        df = pd.read_excel(self.excel_path)

        # Standardize formatting: lowercase types, cleaned product names, and regions
        df = df.assign(
            **{
                'Element Type': df['Element Type'].astype(str).str.lower().str.strip(),
                'Impact filters': df['Impact filters'].astype(str).str.replace(' ', '').str.lower(),
                'Product': df['Product'].astype(str).str.strip()
            }
        )

        # Separate data by type
        self.promotions = df.query("`Element Type` == 'promotion'").to_dict(orient='records')
        self.stockouts = df.query("`Element Type` == 'stockout'").to_dict(orient='records')
        self.distributor_activity = df.query("`Element Type` == 'distributor'").to_dict(orient='records')
        self.events = df.query("`Element Type` in ['seasonal','temperature']").to_dict(orient='records')

        # Unique list of products and regions
        self.products = df['Product'].dropna().unique().tolist()
        self.regions = pd.Series(df['Impact filters'].dropna().map(self.extract_region)).dropna().unique().tolist()

        # Summary of loaded data
        print(f"Loaded: {len(self.products)} products, {len(self.regions)} regions, "
              f"{len(self.promotions)} promotions, {len(self.stockouts)} stockouts, "
              f"{len(self.distributor_activity)} distributor activities, {len(self.events)} events.")

    def extract_region(self, filter_str): 
        """Extract region value from a filter string like 'region=north,product=AC-X1'"""
        for part in filter_str.split(','):
            if 'region=' in part:
                return part.split('=')[1].strip()
        return None

    def format_date_string(self, date_value): 
        """Convert various date formats to a uniform DD-MMM format (e.g., '23-Jul')."""
        if pd.isna(date_value) or date_value is None:
            return None
        if isinstance(date_value, str) and len(date_value.split('-')) == 2:
            return date_value
        if isinstance(date_value, (pd.Timestamp, datetime)):
            return date_value.strftime("%d-%b")
        for fmt in ["%Y-%m-%d", "%d-%m-%Y", "%m/%d/%Y", "%d/%m/%Y"]:
            try:
                return datetime.strptime(str(date_value), fmt).strftime("%d-%b")
            except ValueError:
                continue
        return str(date_value)  # fallback: return raw string

    def get_impact_weeks(self, entry, year=2025): 
        """Calculate the ISO week numbers that a given impact spans."""
        try:
            typ = str(entry.get('Impact Window Type', '')).lower().strip()

            # Handle different types of impact windows
            if typ in ['fixed yearly occurrence', 'from event']:
                s, e = entry.get('Impact Window Start Fixed'), entry.get('Impact Window End Fixed')
                s, e = self.format_date_string(s), self.format_date_string(e)

            elif typ == 'relative':
                base = self.format_date_string(entry.get('Impact Window Start Fixed'))
                if not base:
                    return []
                so = int(entry.get('Impact Window Start', 0) or 0)
                eo = int(entry.get('Impact Window End', 0) or 0)
                base_date = datetime.strptime(f"{base}-{year}", "%d-%b-%Y")
                s = (base_date + timedelta(weeks=so)).strftime("%d-%b")
                e = (base_date + timedelta(weeks=eo)).strftime("%d-%b")
            else:
                return []

            # Convert start and end dates to ISO week numbers
            sd = datetime.strptime(f"{s}-{year}", "%d-%b-%Y")
            ed = datetime.strptime(f"{e}-{year}", "%d-%b-%Y")
            sw, ew = sd.isocalendar()[1], ed.isocalendar()[1]

            # Handle wrapping over year-end if needed
            return list(range(sw, ew + 1 + (52 if ew < sw else 0)))
        except:
            return []

    def match_entry(self, entry, product_code, region, week, year=2025): 
        """Check if a given entry matches the product, region, and week."""
        try:
            # Parse impact filters to dictionary
            filters = dict(item.split('=') for item in str(entry.get('Impact filters', '')).split(',') if '=' in item)

            # Check if product or item_group matches
            p_match = filters.get('product', '').lower() == product_code.lower() or \
                      filters.get('item_group', '').lower() == product_code.lower()
            r_match = filters.get('region', '').lower() == region.lower()

            # Check if the given week falls in the impact window
            w_match = week in self.get_impact_weeks(entry, year)

            return p_match and r_match and w_match
        except:
            return False

    def get_context(self, product_code, region, week, year=2025): 
        """Return the matched context entries for a product, region, and week."""
        return {
            'product_code': product_code,
            'region': region,
            'week': week,
            'promotions': [p for p in self.promotions if self.match_entry(p, product_code, region, week, year)],
            'stockouts': [s for s in self.stockouts if self.match_entry(s, product_code, region, week, year)],
            'events': [e for e in self.events if self.match_entry(e, product_code, region, week, year)],
            'distributor_activity': next(
                (d for d in self.distributor_activity if self.match_entry(d, product_code, region, week, year)), {}
            )
        }


def pandas_to_serializable(obj):
    """Convert pandas/numpy objects into plain JSON-serializable Python data types."""
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        return obj.where(pd.notnull(obj), None).to_dict(orient='records' if isinstance(obj, pd.DataFrame) else None)
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if isinstance(obj, pd.NaT.__class__):
        return None
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: pandas_to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [pandas_to_serializable(v) for v in obj]
    return obj


# Run only if this script is executed directly (not imported)
if __name__ == "__main__":
    # Path to Excel file containing business context data
    excel_path = r"../backend/data/dummy_data.xlsx"

    # Create database object by loading the Excel data
    db = ExcelFrappeDB(excel_path)

    # Fetch context for product 'AC-X1' in 'North' region during ISO week 35 of year 2025
    sample = db.get_context('AC-X1', 'North', 35)

    # Convert context dictionary into a JSON-formatted string and print
    print(json.dumps(pandas_to_serializable(sample), indent=2))


Loaded: 8 products, 3 regions, 2 promotions, 1 stockouts, 1 distributor activities, 4 events.
{
  "product_code": "AC-X1",
  "region": "North",
  "week": 35,
  "promotions": [],
  "stockouts": [],
  "events": [
    {
      "Product": "Back to School - AC-X1 North",
      "Element Type": "seasonal",
      "Event Description": "Increased purchases for electronics in Back to School campaign in North",
      "Impact Window Type": "Fixed Yearly Occurrence",
      "Impact Window Start": NaN,
      "Impact Window End": NaN,
      "Impact Window Start Fixed": "2025-08-25T00:00:00",
      "Impact Window End Fixed": "2025-09-05T00:00:00",
      "Response Type": "Categorical",
      "Impact Magnitude Formula": NaN,
      "Impact filters": "product=ac-x1,region=north",
      "Activity Level": "Low"
    }
  ],
  "distributor_activity": {}
}


In [25]:
# Import necessary modules
import ollama                             # Used to interact with the Ollama LLM
from crewai import Agent, Task, Crew      # CrewAI components for building agent workflows

class OllamaLLM:
    """Wrapper class for using the Ollama LLM with a simple callable interface."""
    def __init__(self, model_name='llama3.2'):
        # Store the model name to be used
        self.model_name = model_name

    def __call__(self, prompt: str) -> str:
        # When called, send the prompt to the Ollama model and return the response
        try:
            return ollama.chat(
                model=self.model_name,
                messages=[{'role': 'user', 'content': prompt}]
            )['message']['content']
        except Exception as e:
            # Handle and return any errors during model call
            return f"LLM Error: {e}"

class ForecastingPipeline:
    """Main pipeline class that executes the end-to-end forecasting enhancement workflow."""

    def __init__(self, mock_db):
        self.mock_db = mock_db                        # Data source (mock or real DB)
        self.llm = OllamaLLM()                        # Initialize LLM wrapper
        self.context_tool = ContextFetcherTool()      # Tool to fetch business context
        self.calc_tool = ForecastCalculatorTool()     # Tool to perform forecast calculations
        self._init_agents_tasks_crew()                # Setup agents, tasks, and crew

    def _init_agents_tasks_crew(self):
        """Define agents, tasks, and the crew process for sequential workflow."""

        # Agent 1: Responsible for gathering context data like events and promotions
        self.context_fetcher = Agent(
            role='Business Context Analyst',
            goal='Gather promotions, stockouts, distributor activities, and events',
            backstory='Expert in retail operations and supply chain',
            tools=[self.context_tool],
            llm=self.llm
        )

        # Agent 2: Uses the context to enhance the forecast
        self.forecast_enhancer = Agent(
            role='Demand Forecasting Specialist',
            goal='Adjust baseline forecasts using business context',
            backstory='15+ years in retail forecasting',
            tools=[self.calc_tool],
            llm=self.llm
        )

        # Agent 3: Explains the results in a business-friendly way
        self.explanation_agent = Agent(
            role='Business Intelligence Communicator',
            goal='Translate forecasting results into actionable recommendations',
            backstory='Expert in translating complex data into actionable insights',
            llm=self.llm
        )

        # Task 1: Context fetching task assigned to the context_fetcher agent
        self.context_task = Task(
            description=("Fetch business context for:\nProduct: {product_code}\nRegion: {region}\nWeek: {week}\n"
                         "Include promotions, stockouts, distributor activity, and events."),
            agent=self.context_fetcher,
            expected_output="Summary of business context with events and multipliers."
        )

        # Task 2: Forecast enhancement task using context summary from Task 1
        self.forecast_task = Task(
            description=("Enhance baseline forecast using context:\nBase Forecast: {base_forecast}\n"
                         "Context: {context_summary}\n"
                         "Calculate adjusted forecast, applied multipliers, and reasoning."),
            agent=self.forecast_enhancer,
            expected_output="Enhanced forecast, multiplier list, and reasoning.",
            context=[self.context_task]  # Uses output from Task 1
        )

        # Task 3: Explanation of the enhanced forecast for business decision-making
        self.explanation_task = Task(
            description=("Explain the adjustment:\nOriginal: {base_forecast}, Enhanced: {enhanced_forecast}\n"
                         "Context: {context_summary}\nProvide summary, key drivers, confidence, and recommendations."),
            agent=self.explanation_agent,
            expected_output="Summary, drivers, confidence level, and recommendations.",
            context=[self.context_task, self.forecast_task]  # Uses outputs from both Task 1 and Task 2
        )

        # Crew ties all agents and tasks together in a sequential process
        self.crew = Crew(
            agents=[self.context_fetcher, self.forecast_enhancer, self.explanation_agent],
            tasks=[self.context_task, self.forecast_task, self.explanation_task],
            verbose=True,       # Prints logs during execution
            process="sequential"  # Tasks are run one after another
        )

    def run_forecast_enhancement(self, product_code, region, week, base_forecast):
        """Run the entire forecasting enhancement pipeline and return results."""
        inputs = {
            'product_code': product_code,
            'region': region,
            'week': week,
            'base_forecast': base_forecast
        }

        try:
            # Start the crew process and return successful output
            result = self.crew.kickoff(inputs)
            return {
                'success': True,
                'product_code': product_code,
                'region': region,
                'week': week,
                'base_forecast': base_forecast,
                'result': result
            }
        except Exception as e:
            # Return error details if the process fails
            return {
                'success': False,
                'error': str(e),
                'inputs': inputs
            }

# Confirmation message to show setup is done
print("Components ready: Agents, Tasks, Crew configured.")


Components ready: Agents, Tasks, Crew configured.


In [27]:
def create_forecast_tasks(product_code: str, region: str, week: int, base_forecast: float,
                          context_fetcher_agent, forecast_enhancer_agent, explanation_agent) -> list:
    """
    Create and return a list of forecasting-related tasks, each tied to a specific agent.
    
    Parameters:
    - product_code (str): Identifier for the product.
    - region (str): Target sales or distribution region.
    - week (int): ISO week number for which forecast is to be made.
    - base_forecast (float): Initial baseline forecast value.
    - context_fetcher_agent: Agent responsible for retrieving business context.
    - forecast_enhancer_agent: Agent responsible for adjusting the forecast.
    - explanation_agent: Agent responsible for explaining the final forecast.

    Returns:
    - List of 3 Task objects in sequence: context retrieval, enhancement, and explanation.
    """

    # Task 1: Retrieve detailed business context for the product-region-week combination
    context_task = Task(
        description=f"""
        Gather comprehensive context for:
        - Product: {product_code}
        - Region: {region}
        - Week: {week}
        Covering:
        1. Promotions and discounts
        2. Stockout situations
        3. Distributor activity
        4. Seasonal or market events
        """,
        agent=context_fetcher_agent,
        expected_output="Detailed business context affecting demand."
    )

    # Task 2: Use context to enhance the forecast with appropriate multipliers
    enhancement_task = Task(
        description=f"""
        Enhance baseline forecast ({base_forecast}) based on context.
        Consider:
        - Promotions (+15–40%), Stockouts (+10–25%), Distributor activity (+10–20%), Seasonal events (+10–50%)
        - Apply cumulatively with diminishing returns.
        Output:
        1. Adjusted forecast
        2. Percentage change
        3. Key adjustment factors
        """,
        agent=forecast_enhancer_agent,
        expected_output="Adjusted forecast with percentages and key factors.",
        context=[context_task]  # Depends on output of the context_task
    )

    # Task 3: Explain the reasoning behind the forecast enhancement in a business-friendly format
    explanation_task = Task(
        description=f"""
        Explain the adjustment clearly for stakeholders:
        - Executive Summary (2–3 sentences)
        - Key Business Drivers (bullet points)
        - Risk Factors
        - Recommended Actions
        - Confidence Level
        """,
        agent=explanation_agent,
        expected_output="Business-friendly explanation with actionable insights.",
        context=[context_task, enhancement_task]  # Depends on both prior tasks
    )

    # Return the task list in correct execution order
    return [context_task, enhancement_task, explanation_task]

# Example usage:
# Uncomment the below line to test task creation
# tasks = create_forecast_tasks("AC-X1", "North", 35, 200, context_fetcher_agent, forecast_enhancer_agent, explanation_agent)
# print(f"Created {len(tasks)} tasks for forecasting.")


In [29]:
from crewai.tools import BaseTool  # Importing the base class for creating custom tools in CrewAI

class ForecastCalculatorTool(BaseTool):
    """Tool for calculating an adjusted forecast based on multiple adjustment factors."""
    
    # Metadata for the tool
    name: str = "Forecast Calculator"
    description: str = "Calculates adjusted forecasts using context factors (promotions, stockouts, activity, events)."

    def _run(self, base_forecast: float, adjustments: list) -> dict:
        """
        Apply a series of percentage-based adjustments (as decimals) to a base forecast.
        
        Parameters:
        - base_forecast: The initial forecast value (e.g., 200 units)
        - adjustments: A list of decimal multipliers (e.g., [0.1, 0.2] for +10% and +20%)

        Returns:
        - Dictionary with adjusted forecast and percentage change from base forecast
        """
        try:
            multiplier = 1.0
            for adj in adjustments:
                # Apply each adjustment cumulatively
                multiplier *= (1 + adj)
            adjusted = round(base_forecast * multiplier)
            pct_change = round((adjusted - base_forecast) / base_forecast * 100, 2)
            return {"adjusted_forecast": adjusted, "percentage_change": pct_change}
        except Exception as e:
            # Return error info in case of failure
            return {"error": str(e)}


class ContextFetcherTool(BaseTool):
    """Tool to fetch and neatly format business context for a given product, region, and week."""
    
    # Metadata for the tool
    name: str = "Business Context Fetcher"
    description: str = "Retrieves relevant context such as promotions, stockouts, distributor activity, and events."

    def __init__(self, mock_db=None, **kwargs):
        # Initialize tool and optionally assign a global mock database reference
        super().__init__(**kwargs)
        if mock_db is not None:
            global global_mock_db
            global_mock_db = mock_db  # Accessible in _run for fetching context

    def _run(self, product_code: str, region: str, week: int) -> str:
        """
        Retrieves and formats business context into a readable summary string.
        
        Parameters:
        - product_code: The code of the product (e.g., 'AC-X1')
        - region: The region (e.g., 'North')
        - week: ISO week number (e.g., 35)

        Returns:
        - A formatted string summarizing all relevant business context
        """
        try:
            # Get business context for given parameters
            ctx = global_mock_db.get_context(product_code, region, week)

            # Start assembling the summary lines
            lines = [f"BUSINESS CONTEXT for {product_code} in {region}, Week {week}"]

            # Section: Promotions
            lines.append("\nPROMOTIONS:")
            lines.extend(
                [f"- {p.get('promotion_type','Unknown')}: {p.get('discount_percent',0)}% discount"
                 for p in ctx.get('promotions', [])] or ["- No active promotions"]
            )

            # Section: Stockouts
            lines.append("\nSTOCKOUTS:")
            lines.extend(
                [f"- Out of stock for {s.get('days_out_of_stock','Unknown')} days"
                 for s in ctx.get('stockouts', [])] or ["- No stockouts reported"]
            )

            # Section: Distributor Activity
            lines.append("\nDISTRIBUTOR ACTIVITY:")
            activity = ctx.get('distributor_activity') or {}
            lines.append(f"- Activity Level: {activity.get('activity_level','N/A')}")
            lines.append(f"- New Distributors: {activity.get('new_distributors','N/A')}")
            lines.append(f"- Performance Score: {activity.get('distributor_performance','N/A')}")

            # Section: Seasonal Events
            lines.append("\nSEASONAL EVENTS:")
            lines.extend(
                [f"- {e.get('event_name','Unknown Event')}: {e.get('impact_level','N/A')} impact"
                 for e in ctx.get('events', [])] or ["- No seasonal events"]
            )

            # Join and return all formatted lines as a string
            return '\n'.join(lines)

        except Exception as e:
            # Return error message if any issue occurs
            return f"Error fetching context: {e}"


In [31]:
# Define a simple mock database to return empty context (used for testing purposes)
class MockDatabase:
    def get_context(self, product_code, region, week):
        """Default empty context for testing"""
        return {
            "promotions": [],
            "stockouts": [],
            "distributor_activity": {},
            "events": []
        }

# Initialize mock_db only if it hasn't been defined elsewhere already
try:
    mock_db
except NameError:
    mock_db = MockDatabase()

# Wrapper class to interact with the Ollama LLM
class OllamaLLM:
    """LLM Wrapper for Ollama model."""
    def __init__(self, model_name='llama3.2'):
        self.model_name = model_name

    def __call__(self, prompt: str) -> str:
        """Send a prompt to the Ollama model and return the response text."""
        try:
            import ollama
            response = ollama.chat(model=self.model_name,
                                    messages=[{'role': 'user', 'content': prompt}])
            return response['message']['content']
        except Exception as e:
            # Return an error message if the model fails to respond
            return f"LLM Error: {e}"

# Instantiate the LLM for use
llm = OllamaLLM()  

# Minimal version of the ContextFetcherTool
class ContextFetcherTool:
    def __init__(self, mock_db):
        self.mock_db = mock_db

    def _run(self, product_code, region, week):
        # Fetch context and return it as a string (simplified)
        context = self.mock_db.get_context(product_code, region, week)
        return str(context)

# Minimal version of the ForecastCalculatorTool
class ForecastCalculatorTool:
    def _run(self, base_forecast, adjustments):
        """
        Simple forecast adjustment: multiply base forecast by (1 + sum of adjustments).
        Example: base_forecast=200, adjustments=[0.25] → 200 * (1 + 0.25) = 250
        """
        adjustment_factor = sum(adjustments) + 1
        return base_forecast * adjustment_factor

# Instantiate both tools with the mock database
context_tool = ContextFetcherTool(mock_db=mock_db)
calc_tool = ForecastCalculatorTool()

# A quick test function to verify basic functionality of the mock components
def quick_test():
    """Run quick sanity checks for context, calculator, and LLM."""
    print("\nQUICK FUNCTIONALITY TEST\n" + "=" * 40)

    # Test 1: Get raw context from mock DB
    context = mock_db.get_context('AC-X1', 'North', 35)
    print(f"1. Context retrieved: {len(context)} fields")

    # Test 2: Format context using tool (stringified output)
    formatted = context_tool._run('AC-X1', 'North', 35)
    print(f"2. Context formatted length: {len(formatted)} chars")

    # Test 3: Run forecast calculator with simple +25% adjustment
    result = calc_tool._run(200, [0.25])
    print(f"3. Calculator result: {result}")

    # Test 4: Run LLM with a simple arithmetic prompt
    try:
        test_response = llm("What is 2 + 2?")
        print(f"4. LLM Response: {test_response[:50]}...")
    except Exception as e:
        print(f"4. LLM Error: {e}")

    print("\nQuick test completed!\n")

# Run the test when this script executes
quick_test()



QUICK FUNCTIONALITY TEST
1. Context retrieved: 4 fields
2. Context formatted length: 77 chars
3. Calculator result: 250.0
4. LLM Response: 2 + 2 = 4...

Quick test completed!



In [33]:
# Mock database with override for get_context to simulate realistic scenarios
class MockDatabase:
    def get_context(self, product_code, region, week):
        """Default empty context for testing"""
        return {
            "promotions": [],
            "stockouts": [],
            "distributor_activity": {},
            "events": []
        }

mock_db = MockDatabase()

# Save original method for optional restoration later
original_get_context = mock_db.get_context

# Override get_context to provide hardcoded test scenarios
def mock_get_context(product, region, week):
    """Mocked business context for demonstration scenarios."""
    if (product, region, week) == ('TV-S2', 'North', 45):
        return {
            'promotions': [{'type': 'Diwali Sale', 'intensity': 'high', 'discount_percent': 35}],
            'events': [{'event_name': 'Diwali', 'impact_level': 'High'}],
            'distributor_activity': {'activity_level': 'very_high'}
        }
    if (product, region, week) == ('REF-C3', 'East', 15):
        return {
            'promotions': [{'type': 'Regular Promo', 'intensity': 'low', 'discount_percent': 10}],
            'events': [],
            'distributor_activity': {'activity_level': 'normal'}
        }
    return {'promotions': [], 'events': [], 'distributor_activity': {}}

# Apply the mock context function to the mock_db instance
mock_db.get_context = mock_get_context

# Main class for simulating forecast adjustment based on context
class MockInterface:
    """Updated forecast adjustment interface with multiplier logic and diminishing returns."""

    def manual_forecast_adjustment(self, product, region, week, base_forecast):
        """Perform manual forecast adjustment based on actual context and multipliers."""
        context = mock_db.get_context(product, region, week)
        multipliers, reasons = self.extract_multipliers_and_reasons(context)

        # Apply each multiplier to base forecast
        adjusted = base_forecast
        for m in multipliers:
            adjusted *= m

        adj_pct = round((adjusted - base_forecast) / base_forecast * 100, 1)
        return round(adjusted), adj_pct, reasons

    def extract_multipliers_and_reasons(self, context):
        """Generate multipliers and reasons based on actual context values."""
        multipliers = []
        reasons = []

        # Process promotions
        for promo in context.get('promotions', []):
            discount_pct = promo.get('discount_percent', 0)
            if discount_pct > 0:
                multiplier = 1 + discount_pct / 100
                multipliers.append(multiplier)
                reasons.append(f"Promotion ({discount_pct:.0f}%): +{discount_pct:.0f}%")

        # Process stockouts
        for stockout in context.get('stockouts', []):
            days_out = stockout.get('days_out_of_stock', 0)
            if days_out >= 3:
                if days_out < 10:
                    multiplier = 1.10
                    reasons.append(f"Short stockout ({days_out} days): +10%")
                else:
                    multiplier = 1.20
                    reasons.append(f"Long stockout ({days_out} days): +20%")
                multipliers.append(multiplier)

        # Process distributor activity levels
        activity = context.get('distributor_activity') or {}
        level = activity.get('activity_level', '').lower()
        mapping = {'low': 1.05, 'normal': 1.10, 'high': 1.15, 'very_high': 1.20}
        mult = mapping.get(level)
        if mult:
            multipliers.append(mult)
            reasons.append(f"Distributor activity ({level.replace('_', ' ').title()}): +{(mult-1)*100:.0f}%")

        # Process events
        for event in context.get('events', []):
            impact = event.get('impact_level', '').lower()
            emap = {'low': 1.05, 'medium': 1.10, 'high': 1.20}
            mult = emap.get(impact)
            if mult:
                multipliers.append(mult)
                reasons.append(f"Event ({event.get('event_name', 'Event')} - {impact.title()}): +{(mult-1)*100:.0f}%")

        # Apply diminishing returns if more than 2 factors
        if len(multipliers) > 2:
            for i in range(2, len(multipliers)):
                original = multipliers[i]
                multipliers[i] = 1 + (original - 1) * 0.5
                reasons[i] += " (diminishing returns)"

        return multipliers, reasons

# Instantiate the forecast adjustment interface
interface = MockInterface()

# Demonstrates the complete forecast enhancement workflow for predefined scenarios
def demonstrate_complete_workflow():
    """Run and display end-to-end examples of forecast enhancement."""
    scenarios = [
        ("High-Impact Scenario", 'TV-S2', 'North', 45, 150),
        ("Low-Impact Scenario", 'REF-C3', 'East', 15, 80),
    ]
    for title, product, region, week, base in scenarios:
        print(f"\n{title}\n" + "-" * 40)
        context = mock_db.get_context(product, region, week)
        print(f"Product: {product}, Region: {region}, Week: {week}, Base Forecast: {base}")
        print(f"Business Context: "
              f"Promotions={len(context.get('promotions', []))}, "
              f"Events={len(context.get('events', []))}, "
              f"Distributor Activity={context.get('distributor_activity', {}).get('activity_level', 'N/A')}")

        enhanced, adj_pct, reasons = interface.manual_forecast_adjustment(product, region, week, base)
        print(f"Enhanced Forecast: {enhanced} units ({adj_pct:+.1f}%), Factors: {len(reasons)}")
        for reason in reasons:
            print(f"  - {reason}")

# Factory function to create a batch forecast processor for handling multiple products/requests
def create_batch_forecast_processor():
    """Batch processor for forecast requests."""
    class BatchForecastProcessor:
        def __init__(self, interface):
            self.interface = interface
            self.results = []

        def process_batch(self, requests):
            """Run forecast adjustments for a list of requests and store results."""
            for req in requests:
                enhanced, adj_pct, reasons = self.interface.manual_forecast_adjustment(
                    req['product'], req['region'], req['week'], req['base_forecast']
                )
                self.results.append({
                    **req,
                    'enhanced_forecast': enhanced,
                    'adjustment_percentage': adj_pct,
                    'adjustment_reasons': reasons,
                    'confidence': 'High' if len(reasons) >= 2 else 'Medium'
                })
            return self.results

        def export_results(self):
            """Export results as a pandas DataFrame."""
            import pandas as pd
            return pd.DataFrame(self.results)

    return BatchForecastProcessor(interface)

# Prints an overview of system architecture and logic
def create_summary_report():
    """Summary of the forecasting system components."""
    print("\nSYSTEM SUMMARY REPORT\n" + "=" * 60)
    print("\nARCHITECTURE:")
    print("• Mock Frappe Database: Simulates business context storage")
    print("• Context Fetcher Tool: Retrieves relevant business data")
    print("• Agents:")
    print("  - Context Fetcher Agent: Analyzes business environment")
    print("  - Forecast Enhancer Agent: Adjusts forecasts based on context")
    print("  - Explanation Generator Agent: Provides actionable business insights")
    print("• Factors Considered:")
    print("  - Promotions (10–40%), Stockouts (10–30%), Distributor Activity (10–20%), Seasonal Events (10–50%)")
    print("  - Diminishing Returns Logic implemented.\n")

# Execute demonstration and system summary
print("\nRUNNING DEMONSTRATION...\n" + "=" * 30)
demonstrate_complete_workflow()
batch_processor = create_batch_forecast_processor()
print(f"\nBatch processor created: {type(batch_processor).__name__}")
create_summary_report()

# Restore original context method to avoid interference elsewhere
mock_db.get_context = original_get_context



RUNNING DEMONSTRATION...

High-Impact Scenario
----------------------------------------
Product: TV-S2, Region: North, Week: 45, Base Forecast: 150
Business Context: Promotions=1, Events=1, Distributor Activity=very_high
Enhanced Forecast: 267 units (+78.2%), Factors: 3
  - Promotion (35%): +35%
  - Distributor activity (Very High): +20%
  - Event (Diwali - High): +20% (diminishing returns)

Low-Impact Scenario
----------------------------------------
Product: REF-C3, Region: East, Week: 15, Base Forecast: 80
Business Context: Promotions=1, Events=0, Distributor Activity=normal
Enhanced Forecast: 97 units (+21.0%), Factors: 2
  - Promotion (10%): +10%
  - Distributor activity (Normal): +10%

Batch processor created: BatchForecastProcessor

SYSTEM SUMMARY REPORT

ARCHITECTURE:
• Mock Frappe Database: Simulates business context storage
• Context Fetcher Tool: Retrieves relevant business data
• Agents:
  - Context Fetcher Agent: Analyzes business environment
  - Forecast Enhancer Agent: 

In [35]:
class ForecastEvaluator:
    """Evaluate forecast accuracy using standard metrics."""

    def __init__(self):
        # Initialize an empty list to store forecast result records
        self.records = []

    def add_result(self, product, region, week, base_forecast, enhanced_forecast, actual):
        """Store a single forecast result for evaluation."""
        self.records.append({
            'product': product,
            'region': region,
            'week': week,
            'base_forecast': base_forecast,         # Original forecast before adjustment
            'enhanced_forecast': enhanced_forecast, # Forecast after adjustment using context
            'actual': actual                        # Actual observed demand/sales
        })

    def evaluate(self):
        """Compute MAPE and RMSE for base and enhanced forecasts."""
        df = pd.DataFrame(self.records)

        # Handle case when no records are available
        if df.empty:
            return {"error": "No records to evaluate."}

        # Calculate Mean Absolute Percentage Error (MAPE) for both forecasts
        df['base_error_pct'] = np.abs(df['actual'] - df['base_forecast']) / df['actual'] * 100
        df['enhanced_error_pct'] = np.abs(df['actual'] - df['enhanced_forecast']) / df['actual'] * 100

        # Calculate squared errors for both forecasts
        df['base_sq_error'] = (df['actual'] - df['base_forecast'])**2
        df['enhanced_sq_error'] = (df['actual'] - df['enhanced_forecast'])**2

        # Compute average MAPE values
        mape_base = df['base_error_pct'].mean()
        mape_enhanced = df['enhanced_error_pct'].mean()

        # Compute RMSE (Root Mean Square Error) values
        rmse_base = np.sqrt(df['base_sq_error'].mean())
        rmse_enhanced = np.sqrt(df['enhanced_sq_error'].mean())

        # Return metrics along with detailed DataFrame
        return {
            'MAPE (Base Forecast)': round(mape_base, 2),
            'MAPE (Enhanced Forecast)': round(mape_enhanced, 2),
            'RMSE (Base Forecast)': round(rmse_base, 2),
            'RMSE (Enhanced Forecast)': round(rmse_enhanced, 2),
            'Details': df  # Optional: full breakdown of forecast vs actual
        }

# Create evaluator instance
evaluator = ForecastEvaluator()

# Add test results (can be replaced with real data)
evaluator.add_result('TV-S2', 'North', 45, 150, 267, actual=280)  # Diwali promotion case
evaluator.add_result('REF-C3', 'East', 15, 80, 97, actual=95)     # Regular case

# Run the evaluation
results = evaluator.evaluate()

# Print summarized evaluation metrics
print("\nEVALUATION METRICS")
for k, v in results.items():
    if k != 'Details':
        print(f"{k}: {v}")

# Print detailed DataFrame of evaluation
print("\nDetailed results:")
print(results['Details'])



EVALUATION METRICS
MAPE (Base Forecast): 31.11
MAPE (Enhanced Forecast): 3.37
RMSE (Base Forecast): 92.53
RMSE (Enhanced Forecast): 9.3

Detailed results:
  product region  week  base_forecast  enhanced_forecast  actual  \
0   TV-S2  North    45            150                267     280   
1  REF-C3   East    15             80                 97      95   

   base_error_pct  enhanced_error_pct  base_sq_error  enhanced_sq_error  
0       46.428571            4.642857          16900                169  
1       15.789474            2.105263            225                  4  


In [37]:
import gradio as gr

# Initialize the forecasting interface and evaluator
interface = MockInterface()
evaluator = ForecastEvaluator()

def forecast_and_evaluate(product, region, week, base_forecast, actual):
    """Run forecast adjustment and compute evaluation metrics."""

    # Apply context-based forecast adjustment using the MockInterface
    enhanced, adj_pct, reasons = interface.manual_forecast_adjustment(product, region, week, base_forecast)

    # Store this forecast result for future evaluation
    evaluator.add_result(product, region, week, base_forecast, enhanced, actual)

    # Run evaluation to calculate accuracy metrics
    results = evaluator.evaluate()

    # Prepare summary output text for display
    out_text = f"""
**Forecast Summary**
- Base Forecast: {base_forecast}
- Enhanced Forecast: {enhanced} (+{adj_pct:.1f}%)
- Actual: {actual}

**Reasons for Adjustment**
""" + "\n".join([f"- {r}" for r in reasons])  # List all factors contributing to the adjustment

    # Prepare metrics output text for display
    metrics_text = f"""
**Evaluation Metrics**
- MAPE (Base): {results['MAPE (Base Forecast)']}%
- MAPE (Enhanced): {results['MAPE (Enhanced Forecast)']}%
- RMSE (Base): {results['RMSE (Base Forecast)']}
- RMSE (Enhanced): {results['RMSE (Enhanced Forecast)']}
"""

    return out_text, metrics_text


# Define the Gradio UI layout
with gr.Blocks() as demo:
    gr.Markdown("# Forecast Enhancement Interface")  # UI header

    # Row for entering product and region
    with gr.Row():
        product = gr.Textbox(label="Product Code", value="TV-S2")  # Default product
        region = gr.Textbox(label="Region", value="North")         # Default region

    # Row for entering week number, base forecast, and actual sales
    with gr.Row():
        week = gr.Number(label="Week Number", value=45)
        base_forecast = gr.Number(label="Base Forecast", value=150)
        actual = gr.Number(label="Actual Sales", value=280)

    # Button to trigger forecast enhancement and evaluation
    run_btn = gr.Button("Run Forecast & Evaluate")

    # Output areas to display forecast summary and evaluation metrics
    summary_out = gr.Markdown()
    metrics_out = gr.Markdown()

    # When the button is clicked, run the function and update the output components
    run_btn.click(
        forecast_and_evaluate,
        inputs=[product, region, week, base_forecast, actual],
        outputs=[summary_out, metrics_out]
    )

# Launch the Gradio app in the browser
demo.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
